# Soup take-home runner

I kept this notebook self-contained on purpose. It creates the small repo under `/content/soup-takehome`, installs Soup in its own Python 3.12 venv, builds a 500-row preference dataset, runs the Soup checks, starts training, records raw GPU samples, then checks whether an adapter was actually produced.

The important part is the verdict logic. A setup banner is not training. I only trust the run if I see GPU activity, progress logs, saved adapter files, and a verification check that can tell "trained something" from "nothing changed".


## What I am checking

The task asks for DPO with layer streaming on a Colab T4, then a clear `SHIP` or `DON'T SHIP` call. I am treating this as an evidence problem, not a leaderboard problem.

I will keep the failed outputs too. If Soup says the setup is fine but the GPU stays idle and no adapter appears, that is useful evidence.


## How I will decide if it did not train

The failure I care about is a quiet one: the command finishes or sits there, but the model never really changes. So I am not treating the setup banner as proof of training.

For this run, these signs mean `DON'T SHIP`:

- `soup train` prints the training setup but never prints real step progress.
- `nvidia-smi` stays near `3 MiB` and `0%` utilization, which means the model is not loaded on the T4.
- The output directory exists but contains no adapter files.
- The verification script cannot find a persisted adapter and cannot measure a behavior shift.

If that happens, the honest conclusion is not "the model trained badly". It is stronger than that: I do not have evidence that training started at all. The next thing to debug would be the handoff between Soup's setup phase and the actual Transformers/DPO loop: dataset loading, model download, layer streaming start, and whether the process is waiting silently before the first batch.


In [ ]:
# First, record the machine Colab gave me
!date -u +%Y-%m-%dT%H:%M:%SZ
!nvidia-smi
import sys, platform, torch
print('python', sys.version)
print('platform', platform.platform())
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('vram_gib', torch.cuda.get_device_properties(0).total_memory / 1024**3)


In [ ]:
# Write the tiny project into /content so the run is reproducible
from pathlib import Path
ROOT = Path('/content/soup-takehome')
for d in ['configs', 'scripts', 'data', 'logs', 'reports', 'output', 'verification']:
    (ROOT / d).mkdir(parents=True, exist_ok=True)
(ROOT / 'scripts/create_dataset.py').write_text('import argparse\nimport hashlib\nimport json\nimport random\nfrom pathlib import Path\n\nfrom datasets import load_dataset\n\n\ndef text(row, *names):\n    """Pick the first non-empty field from a few common preference-dataset names."""\n    for name in names:\n        value = row.get(name)\n        if isinstance(value, str) and value.strip():\n            return value.strip()\n    return ""\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--dataset", default="eridai/russian_dpo_qa")\n    parser.add_argument("--split", default="train")\n    parser.add_argument("--output", default="data/russian_dpo_500.jsonl")\n    parser.add_argument("--n", type=int, default=500)\n    parser.add_argument("--seed", type=int, default=42)\n    args = parser.parse_args()\n\n    ds = load_dataset(args.dataset, split=args.split)\n    rng = random.Random(args.seed)\n    indices = rng.sample(range(len(ds)), min(args.n, len(ds)))\n\n    out = Path(args.output)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    kept = 0\n    with out.open("w", encoding="utf-8") as f:\n        for i in indices:\n            row = dict(ds[int(i)])\n            prompt = text(row, "prompt", "question", "input", "instruction")\n            chosen = text(row, "chosen", "accepted", "response_j", "answer")\n            rejected = text(row, "rejected", "rejected_response", "response_k")\n            # Some public datasets have extra columns or a few odd rows. I keep only\n            # rows that look like plain DPO triples so Soup gets one predictable shape.\n            if not (prompt and chosen and rejected):\n                continue\n            f.write(json.dumps(\n                {"prompt": prompt, "chosen": chosen, "rejected": rejected},\n                ensure_ascii=False,\n            ) + "\\n")\n            kept += 1\n\n    digest = hashlib.sha256(out.read_bytes()).hexdigest()\n    print(f"source_dataset={args.dataset}")\n    print(f"source_rows={len(ds)}")\n    print(f"sample_seed={args.seed}")\n    print(f"written_rows={kept}")\n    print(f"output={out}")\n    print(f"sha256={digest}")\n    if kept < args.n:\n        raise SystemExit(f"only wrote {kept} rows; expected {args.n}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(ROOT / 'scripts/verify_training.py').write_text('import argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\n\ndef sha256(path):\n    """Hash artifacts so the report can point to exact files, not hand-wavy output."""\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef load_pairs(path, limit):\n    rows = []\n    with open(path, "r", encoding="utf-8") as f:\n        for line in f:\n            if line.strip():\n                rows.append(json.loads(line))\n            if len(rows) >= limit:\n                break\n    return rows\n\n\ndef score_answer(model, tokenizer, prompt, answer, max_length):\n    import torch\n\n    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).input_ids\n    full_ids = tokenizer(\n        prompt + answer,\n        return_tensors="pt",\n        truncation=True,\n        max_length=max_length,\n    ).input_ids\n    # Score only the answer tokens. The prompt is context, not something the\n    # adapter should get credit for predicting.\n    answer_start = min(prompt_ids.shape[1], full_ids.shape[1] - 1)\n    full_ids = full_ids.to(model.device)\n    with torch.no_grad():\n        logits = model(full_ids).logits[:, :-1, :]\n        labels = full_ids[:, 1:]\n        logp = torch.log_softmax(logits, dim=-1)\n        token_logp = logp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)\n    return float(token_logp[:, answer_start - 1 :].mean().detach().cpu())\n\n\ndef preference_margin_probe(base_model, adapter_dir, data_path, limit, max_length):\n    import torch\n    from peft import PeftModel\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\n\n    # This is intentionally small. I only need to catch "adapter did nothing",\n    # not run a full eval suite inside the take-home notebook.\n    quant = BitsAndBytesConfig(load_in_4bit=True)\n    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    model = AutoModelForCausalLM.from_pretrained(\n        base_model,\n        quantization_config=quant,\n        device_map="auto",\n        trust_remote_code=True,\n    )\n    pairs = load_pairs(data_path, limit)\n\n    def margins(m):\n        values = []\n        for row in pairs:\n            chosen = score_answer(m, tokenizer, row["prompt"], row["chosen"], max_length)\n            rejected = score_answer(m, tokenizer, row["prompt"], row["rejected"], max_length)\n            values.append(chosen - rejected)\n        return values\n\n    base_margins = margins(model)\n    model = PeftModel.from_pretrained(model, adapter_dir)\n    adapter_margins = margins(model)\n    del model\n    torch.cuda.empty_cache()\n\n    shifts = [a - b for a, b in zip(adapter_margins, base_margins)]\n    return {\n        "pairs": len(pairs),\n        "base_mean_margin": sum(base_margins) / len(base_margins),\n        "adapter_mean_margin": sum(adapter_margins) / len(adapter_margins),\n        "mean_margin_shift": sum(shifts) / len(shifts),\n        "max_abs_margin_shift": max(abs(x) for x in shifts),\n        "all_shifts": shifts,\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--output-dir", default="output")\n    parser.add_argument("--data", default="data/russian_dpo_500.jsonl")\n    parser.add_argument("--report", default="verification/verify_training.json")\n    parser.add_argument("--base-model", default="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")\n    parser.add_argument("--probe-pairs", type=int, default=4)\n    parser.add_argument("--max-length", type=int, default=1024)\n    parser.add_argument("--skip-margin-probe", action="store_true")\n    args = parser.parse_args()\n\n    out = Path(args.output_dir)\n    data = Path(args.data)\n    report = Path(args.report)\n    report.parent.mkdir(parents=True, exist_ok=True)\n\n    files = sorted(p for p in out.rglob("*") if p.is_file()) if out.exists() else []\n    adapter_files = [\n        p for p in files\n        if p.name in {"adapter_model.safetensors", "adapter_model.bin"}\n        or "adapter" in p.name.lower()\n    ]\n\n    result = {\n        "output_dir_exists": out.exists(),\n        "output_file_count": len(files),\n        "output_files": [str(p) for p in files],\n        "adapter_files": [\n            {"path": str(p), "bytes": p.stat().st_size, "sha256": sha256(p)}\n            for p in adapter_files\n        ],\n        "data_rows": sum(1 for _ in data.open("r", encoding="utf-8")) if data.exists() else None,\n        "data_sha256": sha256(data) if data.exists() else None,\n        "passes_minimum_change_check": False,\n        "margin_probe": None,\n        "margin_probe_error": None,\n        "limitations": [\n            "This detects adapter files and a small behavior shift, not model quality.",\n            "It cannot prove the reference model pass was correct.",\n            "It cannot catch data leakage, bad labels, or reward hacking.",\n            "A non-empty adapter can still be trained on the wrong data/config.",\n        ],\n    }\n\n    result["passes_minimum_change_check"] = (\n        result["output_dir_exists"]\n        and len(adapter_files) > 0\n        and all(item["bytes"] > 1024 for item in result["adapter_files"])\n        and result["data_rows"] == 500\n    )\n\n    if result["passes_minimum_change_check"] and not args.skip_margin_probe:\n        try:\n            result["margin_probe"] = preference_margin_probe(\n                args.base_model,\n                str(out),\n                str(data),\n                args.probe_pairs,\n                args.max_length,\n            )\n            result["passes_minimum_change_check"] = (\n                result["passes_minimum_change_check"]\n                and result["margin_probe"]["max_abs_margin_shift"] > 1e-5\n            )\n        except Exception as exc:\n            result["margin_probe_error"] = repr(exc)\n            result["passes_minimum_change_check"] = False\n\n    report.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")\n    print(json.dumps(result, indent=2, ensure_ascii=False))\n    if not result["passes_minimum_change_check"]:\n        raise SystemExit("verification failed: no convincing adapter artifact or behavior change")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
(ROOT / 'configs/soup.yaml').write_text('base: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B\ntask: dpo\n\ndata:\n  train: ./data/russian_dpo_500.jsonl\n  format: dpo\n  max_length: 4096\n\ntraining:\n  epochs: 1\n  lr: 5e-6\n  batch_size: 1\n\n  lora:\n    r: 16\n    alpha: 32\n    target_modules: auto\n\n  quantization: 4bit\n  dpo_beta: 0.1\n\n  stream_layers: true\n  stream_source: auto\n  stream_buffers: 2\n\n  seed: 42\n\noutput: ./output\n\n', encoding='utf-8')
(ROOT / 'soup.yaml').write_text('base: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B\ntask: dpo\n\ndata:\n  train: ./data/russian_dpo_500.jsonl\n  format: dpo\n  max_length: 4096\n\ntraining:\n  epochs: 1\n  lr: 5e-6\n  batch_size: 1\n\n  lora:\n    r: 16\n    alpha: 32\n    target_modules: auto\n\n  quantization: 4bit\n  dpo_beta: 0.1\n\n  stream_layers: true\n  stream_source: auto\n  stream_buffers: 2\n\n  seed: 42\n\noutput: ./output\n\n', encoding='utf-8')
(ROOT / 'reports/final_report.md').write_text('# Soup AI Engineer Take-Home Report\n\n## Verdict\n\n**DON\'T SHIP.**\n\nThe run reached the training setup screen, but I found no evidence that training started or changed the model. I stopped it after about 27 minutes. The GPU stayed idle, the log showed no progress after setup, the output directory had no adapter files, and my verification script failed.\n\nThe strongest evidence:\n\n- `soup doctor` passed and the T4 was visible.\n- `soup profile` estimated 4.07 GB total memory for the configured run.\n- `soup train` printed `Training Setup` at `2026-09-29T11:05:12Z`.\n- From `11:31:03Z` through `11:32:02Z`, `nvidia-smi` stayed at `3 MiB / 15360 MiB`, `0%` GPU utilization.\n- `output_file_count` was `0`.\n- `adapter_files` was empty.\n- `passes_minimum_change_check` was `false`.\n\nI treat this as a failed run.\n\n## Memory Budget\n\nThe available GPU was a Tesla T4 with 15,360 MiB shown by `nvidia-smi`, reported by PyTorch as about 14.56 GiB.\n\nSoup\'s profile estimate for `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` with 4-bit LoRA DPO, batch size 1, and sequence length 4096:\n\n| Component | Estimate |\n|---|---:|\n| Quantized model weights | 0.75 GB |\n| LoRA adapter | 0.02 GB |\n| Optimizer / adapter states | 0.05 GB |\n| Activations | 1.75 GB |\n| Runtime overhead | 1.50 GB |\n| **Total** | **4.07 GB** |\n\nThis estimate fits inside T4 memory. I still expected extra memory from the CUDA context, allocator fragmentation, temporary logits, dataloader/tokenizer buffers, and DPO policy/reference work. A real run should use several GiB of VRAM. This run stayed at 3 MiB, so the gap is not normal overhead. The model never loaded onto the GPU.\n\n## Did the Model Train?\n\nNo.\n\nLoss would not prove training by itself. A DPO loop can print metrics while using malformed data, update no trainable adapter, skip batches, or save nothing. I used a separate verification script. It checked the dataset row count and hash, scanned the output directory, looked for adapter artifacts, hashed adapter files, and tried a small preference-margin probe when artifacts existed.\n\nThe verification output showed:\n\n```json\n{\n  "output_dir_exists": true,\n  "output_file_count": 0,\n  "adapter_files": [],\n  "data_rows": 500,\n  "passes_minimum_change_check": false\n}\n```\n\nThat check would not prove quality if it passed. It would miss bad labels, data leakage, a wrong reference-model pass, and overfitting. In this run it caught a simpler failure: Soup wrote no adapter, so there was no trained model to ship.\n\n## Silent Failures\n\n### Environment mismatch\n\n`soup doctor` warned that `soup` used `/content/soup-env/bin/python`, while `python` on PATH pointed to `/usr/local/bin/python`. I used the soup interpreter explicitly for checks and scripts. The required train packages were installed in the soup env: `torch`, `transformers`, `trl`, `peft`, `datasets`, `bitsandbytes`, and `accelerate`.\n\n### Missing preflight command\n\nThe task asked to use pre-flight checks, but `soup-cli==0.75.1` did not expose a `preflight` command:\n\n```text\nNo such command \'preflight\'.\n```\n\nI kept this as raw evidence. I used `soup doctor`, config validation/profile, `soup ship`, and the custom verification script as substitutes.\n\n### Setup banner without training\n\nThe main silent failure: `soup train` printed a valid-looking setup banner and then made no visible progress. The GPU monitor stayed idle for the whole observed window. A setup-only check would miss this.\n\n### Empty output\n\nThe output directory existed, but it had zero files. Soup\'s setup checks did not catch that because the failure happened after train started. My verification script caught it and failed the run.\n\n## What Must Change Before Shipping\n\nI would not ship this run until all of these are true:\n\n- `soup train` logs actual steps, loss/progress, and a completed save.\n- `nvidia_smi_training.csv` shows real GPU activity and memory use.\n- `output/` contains non-empty adapter artifacts.\n- `soup ship` passes after training.\n- `verify_training.py` passes and shows at least a small base-vs-adapter behavior shift.\n- Soup fails loudly if training stalls after setup with no GPU process, no progress, and no artifacts.\n\n## What Surprised Me / What Still Concerns Me\n\nI expected a failed run to throw an error. Instead, the run looked healthy at setup and then sat idle. That worries me because a user can mistake a setup banner for progress.\n\nMy remaining concern: even a valid adapter artifact would not prove the model learned the intended behavior. I would add a held-out preference-margin check, sample generations before/after, and a small regression test that fails if `soup train` exits or stalls without saving an adapter.\n\n## AI Tool Use\n\nI used Codex to draft the Colab runner, verification script, and report structure. I accepted the generated scaffold only after checking the raw logs, `nvidia-smi` output, output directory, and verification result. The final verdict comes from those artifacts, not from the tool\'s suggestion.\n', encoding='utf-8')
%cd /content/soup-takehome
!find . -maxdepth 2 -type f | sort


In [ ]:
# Soup gets its own venv; Colab's default python is a different interpreter
%cd /content/soup-takehome
!sudo apt-get update -qq
!sudo apt-get install -y python3.12-venv >/dev/null
!rm -rf /content/soup-env
!/usr/bin/python3.12 -m venv /content/soup-env
!/content/soup-env/bin/python -m pip install --upgrade pip setuptools wheel -q
!PIP_DEFAULT_TIMEOUT=120 /content/soup-env/bin/python -m pip install --prefer-binary "soup-cli[train,data]==0.75.1"
!/content/soup-env/bin/python -m pip check | tee logs/pip_check.log
!/content/soup-env/bin/python -m pip show soup-cli torch transformers trl peft datasets accelerate bitsandbytes | grep -E '^(Name|Version):' | tee logs/package_versions.log


In [ ]:
# Build the 500 preference pairs and keep the hash for the report
%cd /content/soup-takehome
!/content/soup-env/bin/python scripts/create_dataset.py 2>&1 | tee logs/create_dataset.log
!head -1 data/russian_dpo_500.jsonl
!wc -l data/russian_dpo_500.jsonl
!sha256sum data/russian_dpo_500.jsonl | tee logs/dataset_sha256.log


In [ ]:
# Run Soup's checks before spending time on training
%cd /content/soup-takehome
!set -o pipefail; /content/soup-env/bin/soup doctor --config soup.yaml 2>&1 | tee logs/doctor.log
!set -o pipefail; /content/soup-env/bin/soup preflight --config soup.yaml 2>&1 | tee logs/preflight.log
!set -o pipefail; /content/soup-env/bin/soup profile --config soup.yaml --json 2>&1 | tee reports/profile_before_training.json


In [ ]:
# Start training and sample the GPU every two seconds
%cd /content/soup-takehome
!date -u +%Y-%m-%dT%H:%M:%SZ | tee logs/train_start_utc.txt
!nvidia-smi | tee logs/nvidia_smi_before.txt
!rm -f logs/nvidia_smi_training.csv
!nohup bash -c 'while true; do printf "%s," "$(date -u +%Y-%m-%dT%H:%M:%SZ)"; nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu,power.draw --format=csv,noheader,nounits; sleep 2; done' > logs/nvidia_smi_training.csv 2>&1 & echo $! > logs/nvidia_monitor.pid
!set -o pipefail; /content/soup-env/bin/soup train --config soup.yaml 2>&1 | while IFS= read -r line; do printf '[%s] %s\n' "$(date -u +%Y-%m-%dT%H:%M:%SZ)" "$line"; done | tee logs/train_raw.log
!date -u +%Y-%m-%dT%H:%M:%SZ | tee logs/train_end_utc.txt
!kill $(cat logs/nvidia_monitor.pid) 2>/dev/null || true
!nvidia-smi | tee logs/nvidia_smi_after.txt
!tail -40 logs/train_raw.log
!tail -20 logs/nvidia_smi_training.csv


In [ ]:
# Check shipping, then prove that an adapter exists and changes behavior
%cd /content/soup-takehome
!set -o pipefail; /content/soup-env/bin/soup ship --config soup.yaml 2>&1 | tee logs/ship.log
!find output -maxdepth 4 -type f -printf '%p %s bytes\n' | sort | tee logs/output_files.log
!/content/soup-env/bin/python scripts/verify_training.py 2>&1 | tee logs/verify_training.log
!cat verification/verify_training.json


## How to read the final cells

After the training cell, check three places before writing the report:

1. The raw `nvidia-smi` samples in `logs/nvidia_smi_during_train.csv`. A real run should show much more than `3 MiB` and usually non-zero GPU utilization.
2. `outputs/dpo-run/`. A usable LoRA run should leave adapter files such as `adapter_config.json` and adapter weights.
3. `logs/verify_training.json`. If `passes_minimum_change_check` is `false`, I would not ship the run.

The verification script is deliberately modest. It does not prove semantic quality, label correctness, or that DPO used the reference model correctly. It only answers a smaller but important question: did this run leave behind an adapter and did that adapter measurably change policy-vs-rejected margins on a few examples?


In [ ]:
# Pack the evidence so I can write the final report from raw files
%cd /content/soup-takehome
!tar -czf soup_takehome_artifacts.tgz configs scripts data logs reports output verification soup.yaml
!ls -lh soup_takehome_artifacts.tgz
print('Download /content/soup-takehome/soup_takehome_artifacts.tgz. This is the evidence bundle for the report.')
